In [1]:
# ====================================================
# NeurIPS 2025 - Open Polymer Prediction
# track2.ipynb : Modular Offline Prototype
# ====================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error


In [2]:
# Load datasets (adjust paths for local vs Kaggle later)
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

TARGETS = ["Tg","FFV","Tc","Density","Rg"]

def competition_wmae(y_true_dict, y_pred_dict):
    maes, weights, total_weight = {}, {}, 0.0
    for target in TARGETS:
        y_true_full = np.asarray(y_true_dict[target])
        y_pred_full = np.asarray(y_pred_dict[target])

        mask = (~np.isnan(y_true_full)) & np.isfinite(y_pred_full)
        if mask.sum() == 0: continue

        y_true, y_pred = y_true_full[mask], y_pred_full[mask]
        mae = mean_absolute_error(y_true, y_pred)

        value_range = np.nanmax(y_true) - np.nanmin(y_true)
        if value_range == 0: value_range = 1.0

        weight = (1.0 / np.sqrt(mask.sum())) / value_range
        maes[target] = mae
        weights[target] = weight
        total_weight += weight

    # Normalize weights
    for t in weights:
        weights[t] = weights[t] / total_weight if total_weight > 0 else 0.0

    wmae = sum(maes[t] * weights[t] for t in maes)
    return wmae, maes


In [5]:
import warnings
warnings.filterwarnings("ignore")   # suppress all Python warnings

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')      # suppress RDKit warnings

from rdkit import Chem
from mordred import Calculator, descriptors
import pandas as pd
import numpy as np
from concurrent.futures import ProcessPoolExecutor

# ---------- 1. Parallel SMILES → Mol ----------
def smiles_to_mol(s):
    if not isinstance(s, str): return None
    s2 = s.replace("*","C")  # handle polymer wildcards
    try:
        return Chem.MolFromSmiles(s2)
    except:
        return None

def smiles_to_mols_parallel(smiles_list, max_workers=8):
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        mols = list(executor.map(smiles_to_mol, smiles_list))
    return mols

print("Converting SMILES to Mol objects...")
train_mols = smiles_to_mols_parallel(train["SMILES"].tolist(), max_workers=8)
test_mols  = smiles_to_mols_parallel(test["SMILES"].tolist(), max_workers=8)

# ---------- 2. Mordred Descriptors (Multithreaded) ----------
calc = Calculator(descriptors, ignore_3D=True)

print("Computing Mordred descriptors (train)...")
train_desc = calc.pandas(train_mols, nproc=8)
print("Computing Mordred descriptors (test)...")
test_desc  = calc.pandas(test_mols, nproc=8)

# ---------- 3. Clean Descriptors ----------
train_desc = train_desc.apply(pd.to_numeric, errors="coerce")
test_desc  = test_desc.apply(pd.to_numeric, errors="coerce")

# Drop all-NaN and constant cols (based on train)
train_desc = train_desc.dropna(axis=1, how="all")
nunique = train_desc.nunique(dropna=True)
train_desc = train_desc.loc[:, nunique > 1]

# Align test columns
test_desc = test_desc.reindex(columns=train_desc.columns)

# Median imputation
med = train_desc.median(numeric_only=True)
train_desc = train_desc.fillna(med)
test_desc  = test_desc.fillna(med)

print("✅ Mordred descriptors ready:", train_desc.shape, test_desc.shape)


Converting SMILES to Mol objects...
Computing Mordred descriptors (train)...


100%|██████████| 7973/7973 [06:11<00:00, 21.46it/s]


Computing Mordred descriptors (test)...


100%|██████████| 3/3 [00:00<00:00, 11.71it/s]


✅ Mordred descriptors ready: (7973, 1412) (3, 1412)


In [6]:
import warnings
warnings.filterwarnings("ignore")

from joblib import Parallel, delayed
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

# ---------- 1. Morgan Fingerprints (Count-based) ----------
def morgan_count_fp(mol, radius=2, nBits=1024):
    if mol is None: 
        return np.zeros(nBits, dtype=float)
    fp = AllChem.GetHashedMorganFingerprint(mol, radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    for idx, count in fp.GetNonzeroElements().items():
        arr[idx % nBits] = count
    return arr.astype(float)

# ---------- 2. Basic RDKit Descriptors ----------
def rdkit_basic(mol):
    if mol is None: 
        return np.array([np.nan]*8, dtype=float)
    try:
        return np.array([
            Descriptors.MolWt(mol),
            Descriptors.TPSA(mol),
            Descriptors.MolLogP(mol),
            rdMolDescriptors.CalcNumRings(mol),
            rdMolDescriptors.CalcNumAromaticRings(mol),
            rdMolDescriptors.CalcNumAliphaticRings(mol),
            rdMolDescriptors.CalcNumHBD(mol),
            rdMolDescriptors.CalcNumHBA(mol),
        ], dtype=float)
    except:
        return np.array([np.nan]*8, dtype=float)

# ---------- 3. Parallel Feature Builder ----------
def build_extra_features(mols, n_jobs=8):
    print("Building Morgan fingerprints...")
    X_fp = Parallel(n_jobs=n_jobs)(delayed(morgan_count_fp)(m) for m in mols)
    print("Building RDKit 2D descriptors...")
    X_rd = Parallel(n_jobs=n_jobs)(delayed(rdkit_basic)(m) for m in mols)

    X_fp = np.vstack(X_fp)
    X_rd = np.vstack(X_rd)

    # Impute NaNs in RDKit descriptors
    col_med = np.nanmedian(X_rd, axis=0)
    inds = np.where(np.isnan(X_rd))
    X_rd[inds] = np.take(col_med, inds[1])

    return np.hstack([X_fp, X_rd])

X_train_extra = build_extra_features(train_mols, n_jobs=8)
X_test_extra  = build_extra_features(test_mols, n_jobs=8)

# Combine with Mordred
X_train_all = np.hstack([train_desc.values, X_train_extra])
X_test_all  = np.hstack([test_desc.values,  X_test_extra])

print("✅ Combined feature shapes:", X_train_all.shape, X_test_all.shape)


Building Morgan fingerprints...


[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerator
[15:52:15] DEPRECATION WARNING: please use MorganGenerat

Building RDKit 2D descriptors...
Building Morgan fingerprints...
Building RDKit 2D descriptors...


[15:52:16] DEPRECATION WARNING: please use MorganGenerator
[15:52:16] DEPRECATION WARNING: please use MorganGenerator
[15:52:16] DEPRECATION WARNING: please use MorganGenerator


✅ Combined feature shapes: (7973, 2444) (3, 2444)


In [11]:
from lightgbm import LGBMRegressor
import gc

def train_lgbm_per_target(X, y_dict, X_test, n_splits=5, seed=42):
    preds_test = {}
    oof_preds = {t: np.full(len(y_dict[t]), np.nan) for t in TARGETS}  # prefill with NaNs

    for target in TARGETS:
        print(f"\n🔹 Training target: {target}")

        y_full = y_dict[target].values
        mask = ~np.isnan(y_full)   # only rows with labels
        X_masked = X[mask]
        y = y_full[mask]

        if len(y) == 0:
            print(f"⚠️ Skipping {target} (no training labels)")
            preds_test[target] = np.zeros(len(X_test))
            continue

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        preds_test_target = []

        for fold, (trn_idx, val_idx) in enumerate(kf.split(X_masked, y)):
            print(f"  Fold {fold+1}/{n_splits}...")

            X_tr, X_val = X_masked[trn_idx], X_masked[val_idx]
            y_tr, y_val = y[trn_idx], y[val_idx]

            model = LGBMRegressor(
                objective="regression",
                metric="mae",
                learning_rate=0.05,
                num_leaves=64,
                feature_fraction=0.8,
                bagging_fraction=0.8,
                bagging_freq=5,
                n_estimators=5000,
                random_state=seed,
                n_jobs=-1
            )

            model.fit(
                X_tr, y_tr,
                eval_set=[(X_val, y_val)],
                eval_metric="mae",
                callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=500)]
            )

            # Store OOF predictions only for rows with labels
            oof_preds[target][mask] = np.where(
                np.isnan(oof_preds[target][mask]),
                0.0,  # init with 0 if needed
                oof_preds[target][mask]
            )
            oof_preds[target][mask][val_idx] = model.predict(
                X_val, num_iteration=model.best_iteration_
            )

            preds_test_target.append(
                model.predict(X_test, num_iteration=model.best_iteration_)
            )

            del X_tr, X_val, y_tr, y_val
            gc.collect()

        preds_test[target] = np.mean(preds_test_target, axis=0)

    return oof_preds, preds_test


In [12]:
# Prepare y dict
y_dict = {t: train[t] for t in TARGETS}

# Run baseline LightGBM with CV
oof_preds, test_preds = train_lgbm_per_target(X_train_all, y_dict, X_test_all)

# Evaluate OOF
wmae, maes = competition_wmae(y_dict, oof_preds)
print("\n✅ OOF wMAE:", round(wmae, 5))
print("Per-target MAEs:")
for t, v in maes.items():
    print(f"  {t}: {v:.4f}")



🔹 Training target: Tg
  Fold 1/5...
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=5, subsample_freq=0 will be ignored. Current value: bagging_freq=5
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014902 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142491
[LightGBM] [Info] Number of data points in the train set: 408, numb

In [1]:
# %% Cell — Inspect full directory tree
import os
from pathlib import Path
import datetime

def scan_dir(root="."):
    root = Path(root).resolve()
    print(f"📂 Scanning directory tree from: {root}\n")
    for path, dirs, files in os.walk(root):
        relpath = os.path.relpath(path, root)
        depth = relpath.count(os.sep)
        indent = "  " * depth
        print(f"{indent}📁 {os.path.basename(path) if relpath != '.' else '.'}/")
        for f in files:
            fp = Path(path) / f
            try:
                size = fp.stat().st_size / 1024  # KB
                mtime = datetime.datetime.fromtimestamp(fp.stat().st_mtime)
                print(f"{indent}  ├─ {f}  ({size:.1f} KB, {mtime:%Y-%m-%d %H:%M})")
            except Exception as e:
                print(f"{indent}  ├─ {f}  (error: {e})")

# 👉 Run this from your project root
scan_dir(".")


📂 Scanning directory tree from: /home/rohit/Desktop/NeurIPS

📁 ./
  ├─ track2.ipynb  (1250.4 KB, 2025-08-20 16:04)
  ├─ neurips-preparedataset-predict.ipynb  (191.6 KB, 2025-08-18 09:45)
  ├─ phase2.ipynb  (578.4 KB, 2025-08-20 17:30)
  ├─ phase1.ipynb  (1245.6 KB, 2025-08-20 16:30)
  ├─ phase4.ipynb  (19.0 KB, 2025-08-21 15:01)
  ├─ submission.csv  (0.1 KB, 2025-08-21 13:16)
  ├─ phase1.html  (6354.9 KB, 2025-08-20 14:16)
  ├─ hybridpipeline.ipynb  (51.0 KB, 2025-08-24 08:02)
  ├─ phase5.ipynb  (148.0 KB, 2025-08-21 17:40)
📁 .git/
  ├─ description  (0.1 KB, 2025-08-18 09:40)
  ├─ config  (0.3 KB, 2025-08-18 09:44)
  ├─ index  (3.7 KB, 2025-08-18 09:44)
  ├─ COMMIT_EDITMSG  (0.0 KB, 2025-08-18 09:44)
  ├─ HEAD  (0.0 KB, 2025-08-18 09:41)
  📁 hooks/
    ├─ pre-commit.sample  (1.6 KB, 2025-08-18 09:40)
    ├─ pre-merge-commit.sample  (0.4 KB, 2025-08-18 09:40)
    ├─ applypatch-msg.sample  (0.5 KB, 2025-08-18 09:40)
    ├─ fsmonitor-watchman.sample  (4.5 KB, 2025-08-18 09:40)
    ├─ pre-